# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library, referencing dataset entities by their Croissant `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
meta = dataset.metadata
print(f"Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Version: {meta.version}")
print(f"Published: {meta.datePublished}")
print(f"License: {meta.license}")
print(f"Cite as: {meta.citeAs}\n")

## 2. Data Overview
Review available record sets, their `@id`, associated fields, and column `@id`s.

In [ ]:
# Retrieve the record sets
print('--- Dataset Record Sets ---')
for rs in dataset.metadata.recordSets:
    print(f"@id: {rs['@id']}")
    print(f"  Name:  {rs.get('name', 'N/A')}")
    if 'description' in rs:
        print(f"  Description: {rs['description']}")
    # List all fields
    print("  Fields:")
    for field in rs.get('fields', []):
        print(f"    - @id: {field['@id']} (name: {field.get('name', field['@id'])})")
    # List columns
    print("  Columns:")
    for col in rs.get('columns', []):
        print(f"    - @id: {col['@id']} (name: {col.get('name', col['@id'])})")
    print()

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. All entities (record sets, fields, columns) are referenced by their `@id`.

**Note:** In this dataset, there is a main table called `cr:recordSet:mass-spectrometry-data` with fields by their `@id`, such as `cr:field:accession`, `cr:field:protein_name`, `cr:field:mw`, etc.

In [ ]:
# List all record set @ids:
record_set_ids = [rs['@id'] for rs in dataset.metadata.recordSets]
print('Record sets:', record_set_ids)

# For this dataset, we'll use the main record set:
main_record_set_id = record_set_ids[0]  # e.g. 'cr:recordSet:mass-spectrometry-data'

# Load records for each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df

print(f"Columns for {main_record_set_id}:")
print(dataframes[main_record_set_id].columns.tolist())

dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process the main mass spectrometry record set. We'll:
- Select a numeric field by its `@id` (`cr:field:mw` = molecular weight in kDa)
- Filter for proteins with MW > 50 kDa
- Normalize the MW column
- Group by a categorical field (e.g., `cr:field:protein_name` if available)
- Output head of each

In [ ]:
# Choose field @ids (please refer to the 'Data Overview' output)
numeric_field_id = 'cr:field:mw'  # Molecular Weight
group_field_id = 'cr:field:protein_name'  # Protein Name

df = dataframes[main_record_set_id]

# Ensure the numeric field is float
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

threshold = 50  # kDa
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold} (MW > 50 kDa):")
print(filtered_df[[numeric_field_id, group_field_id]].head())

# Normalize
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized molecular weight for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by field (mean MW by protein name)
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame("mean_MW")
    print(f"\nGrouped data by {group_field_id} (mean MW):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of Molecular Weight (MW) and explore relationships.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for molecular weight
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
plt.xlabel('Molecular Weight (kDa)')
plt.title('Distribution of Protein Molecular Weight')
plt.show()

# If there are at least two numeric columns, show a scatterplot
if 'cr:field:coverage' in df.columns:
    plt.figure(figsize=(6, 5))
    sns.scatterplot(x=df[numeric_field_id], y=df['cr:field:coverage'])
    plt.xlabel('Molecular Weight (kDa)')
    plt.ylabel('Coverage (%)')
    plt.title('MW vs. Coverage')
    plt.show()

## 6. Conclusion
In this notebook, we loaded the Mass Spectrometry dataset using `mlcroissant`, referenced all entities by `@id`, and performed initial EDA such as filtering proteins by molecular weight, normalization, grouping, and visualizing field distributions. This workflow can be extended for deeper bioinformatics analysis or machine learning as needed.